# WS 02 · Chuẩn bị mệnh đề cho gán nhãn yếu

Notebook này tạo bảng duyệt Stage 02 từ KB bất biến. `observable` nghĩa là mệnh đề có thể xác nhận từ **ảnh hoặc caption** SkinCAP. Chỉ morphology được dùng thêm ConceptCLIP; location, symptom và history chỉ dùng text NLI.

> Không sửa `kb_propositions.json`, không dùng LLM, và không chạy phần chốt phiên bản trước khi Trí duyệt CSV.

## 1. Thiết lập môi trường và đường dẫn

Clone/cập nhật nhánh `WS`, mount Drive và xác định KB bất biến cùng vị trí lưu artefact duyệt.

In [ ]:
from pathlib import Path
import os

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/trisle0805/RAD-colab.git"
BRANCH = "WS"
REPO_DIR = Path("/content/RAD-colab")

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}

%cd {REPO_DIR}
assert (REPO_DIR / "kb_builder/outputs/final/kb_propositions.json").exists()

DRIVE_ROOT = Path("/content/drive/MyDrive/Cao học")
KB_PATH = REPO_DIR / "kb_builder/outputs/final/kb_propositions.json"
OUTPUT_DIR = DRIVE_ROOT / "WeakSup/propositions/v1"
REVIEW_CSV = OUTPUT_DIR / "propositions_review.csv"

print(f"KB: {KB_PATH}")
print(f"Review CSV: {REVIEW_CSV}")

## 2. Tạo và kiểm tra bảng đề xuất

Hàm quy tắc tạo một dòng trên mỗi mệnh đề. Các dòng text-only và các tách tự động được đưa lên đầu để ưu tiên duyệt.

In [ ]:
import pandas as pd

from proposition.weak.propositions import build_review_rows, write_review_csv

review_rows = build_review_rows(KB_PATH)
review = pd.DataFrame(review_rows)
assert len(review) == 1436, f"Expected 1436 propositions, found {len(review)}"
assert review["proposition_id"].is_unique

print("Observable proposals:", int(review["auto_observable"].sum()))
print("Image-applicable proposals:", int(review["image_applicable"].sum()))
print("Rows requiring review:", int(review["review_needed"].sum()))
display(review.groupby(["category", "auto_observable"]).size().rename("count").reset_index())
display(review.head(30))

## 3. Xuất CSV để duyệt thủ công

Sửa trực tiếp `final_observable`, `final_sub_hypotheses` và `note` trong CSV trên Drive. Không thay đổi cột nhận dạng hay thứ tự mệnh đề trong KB.

In [ ]:
write_review_csv(review_rows, REVIEW_CSV)
print(f"Wrote {len(review_rows)} rows to: {REVIEW_CSV}")
print("Review priority: review_needed=1, then all text-only history/symptom rows.")

## 4. Dừng để Trí duyệt

Không chạy phần chốt bên dưới cho đến khi Trí đã xem và sửa `propositions_review.csv`, đặc biệt toàn bộ `review_needed=1`. Không dùng LLM để sửa `canonicalDescription` hay tự quyết các dòng cần duyệt.

## 5. Chốt phiên bản sau duyệt

Đặt `APPROVED_BY_TRI = True` chỉ sau khi CSV đã được duyệt thủ công. Hàm sẽ kiểm tra ID, câu hỏi con, số mệnh đề theo bệnh và bảo toàn thứ tự KB trước khi xuất artefact cuối.

In [ ]:
from proposition.weak.propositions import export_final_version

APPROVED_BY_TRI = False
if not APPROVED_BY_TRI:
    raise RuntimeError(
        "Stop: review propositions_review.csv manually, then set APPROVED_BY_TRI = True."
    )

artifacts = export_final_version(
    KB_PATH,
    REVIEW_CSV,
    OUTPUT_DIR,
)
artifacts